# OCI GenAI Service — Workshop Test Notebook

This notebook tests the Agent Memory Workshop flow using **xAI Grok models** via **OCI GenAI Service**.

It runs the same tests as the main workshop (Oracle vector search, tool calling, entity extraction,
context engineering) but uses OCI GenAI instead of local Ollama. Authentication is via OCI CLI profiles.

**Prerequisites:**
- OCI CLI configured (`~/.oci/config` with DEFAULT profile)
- `oci-openai` package installed (`pip install oci-openai`)
- Oracle AI Database running (same as main workshop)

**Models tested:**
- `xai.grok-3-fast` — fast, high quality (recommended)
- `xai.grok-3-mini-fast` — smaller, still capable


## Setup

In [1]:
import os, sys, time, json, uuid, requests
import oracledb
from openai import OpenAI
from oci_openai import OciOpenAI, OciUserPrincipalAuth

# ==================== CONFIGURATION ====================
OCI_PROFILE = "DEFAULT"
OCI_REGION = "us-ashburn-1"
OCI_COMPARTMENT_ID = "ocid1.tenancy.oc1..aaaaaaaaypz3jeouf67rycnobkfaj7zepotuvpoqtqipfai3qs4qw7ok6yna"

# Change this to test different models
MODEL = "xai.grok-3-fast"
# MODEL = "xai.grok-3-mini-fast"

# Oracle connection (same as main workshop)
ORACLE_DSN = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
ORACLE_USER = os.environ.get("ORACLE_USER", "VECTOR")
ORACLE_PASSWORD = os.environ.get("ORACLE_PASSWORD", "VectorPwd_2025")

print(f"Model: {MODEL}")
print(f"Region: {OCI_REGION}")
print(f"Oracle: {ORACLE_USER}@{ORACLE_DSN}")


Model: xai.grok-3-fast
Region: us-ashburn-1
Oracle: VECTOR@localhost:1521/FREEPDB1


## 1. OCI GenAI Client Connection

In [2]:
# Create OCI GenAI client (authenticates via ~/.oci/config)
auth = OciUserPrincipalAuth(profile_name=OCI_PROFILE)
client = OciOpenAI(
    auth=auth,
    region=OCI_REGION,
    compartment_id=OCI_COMPARTMENT_ID
)

# Quick test
t0 = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 2+2? Reply with just the number."}],
    max_tokens=50
)
elapsed = time.time() - t0
print(f"OCI GenAI connected ✓ — {MODEL}")
print(f"Response: {resp.choices[0].message.content.strip()} ({elapsed:.1f}s)")


OCI GenAI connected ✓ — xai.grok-3-fast
Response: 4 (0.6s)


## 2. Oracle AI Database Connection

In [3]:
vector_conn = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN)
vector_conn.clientinfo = "oci-genai-test"
print(f"Oracle connected ✓ — version {vector_conn.version}")


Oracle connected ✓ — version 23.26.1.0.0


## 3. Vector Search (Embeddings + Oracle)

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-mpnet-base-v2")
print("Embedding model loaded ✓")

# Create test vector store
TEST_TABLE = "OCI_GENAI_TEST_VECTORS"
try:
    vector_conn.cursor().execute(f"DROP TABLE {TEST_TABLE} PURGE")
    vector_conn.commit()
except:
    pass

vector_store = OracleVS(
    client=vector_conn, embedding_function=embedding_model,
    table_name=TEST_TABLE, distance_strategy=DistanceStrategy.COSINE
)

test_docs = [
    "Reinforcement learning for robotic manipulation tasks using deep neural networks",
    "Transformer architectures for natural language understanding and generation",
    "Computer vision techniques for autonomous vehicle perception systems",
    "Graph neural networks for molecular property prediction in drug discovery",
    "Federated learning for privacy-preserving distributed model training",
]
test_meta = [{"source": "test", "id": str(i)} for i in range(len(test_docs))]
vector_store.add_texts(texts=test_docs, metadatas=test_meta)
print(f"Inserted {len(test_docs)} documents ✓")

results = vector_store.similarity_search("robot learning", k=2)
print(f"Search 'robot learning' → {len(results)} results ✓")
for r in results:
    print(f"  • {r.page_content[:70]}...")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/paraphrase-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded ✓


Inserted 5 documents ✓
Search 'robot learning' → 2 results ✓
  • Reinforcement learning for robotic manipulation tasks using deep neura...
  • Federated learning for privacy-preserving distributed model training...


## 4. Tool Calling

Testing whether the model correctly:
1. Calls the right tool with valid arguments
2. Processes tool results and generates a response
3. Shows restraint (doesn't call tools for general knowledge questions)


In [5]:
tools = [
    {"type": "function", "function": {
        "name": "search_papers",
        "description": "Search for research papers on a given topic. Use when the user asks to find, look up, or discover papers.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Search query for papers"},
            "max_results": {"type": "integer", "description": "Maximum results to return"}
        }, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "search_tavily",
        "description": "Search the web for current information, news, or recent events.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Web search query"},
            "max_results": {"type": "integer", "description": "Maximum results"}
        }, "required": ["query"]}
    }}
]

SYSTEM_PROMPT = """You are a Research Paper Assistant with access to tools.

## Tool Usage Restraint
Only call tools when the user's request genuinely requires external data you cannot answer
from memory context or general knowledge. For factual questions you already know, answer directly.

When answering:
1. FIRST, use the context provided in the input
2. Use external search tools only if memory context is insufficient
3. Keep responses evidence-based and aligned with retrieved research context
"""

def call_llm(messages, tools=None, tool_choice="auto"):
    kwargs = {"model": MODEL, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

print("Tools and system prompt configured ✓")


Tools and system prompt configured ✓


### 4a. Tool Selection (should call search_papers)

In [6]:
t0 = time.time()
resp = call_llm(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Find papers about reinforcement learning for robotics"}
    ],
    tools=tools
)
elapsed = time.time() - t0
msg = resp.choices[0].message

if msg.tool_calls:
    tc = msg.tool_calls[0]
    args = json.loads(tc.function.arguments)
    print(f"✓ Tool called: {tc.function.name}")
    print(f"  Arguments: {args}")
    print(f"  Time: {elapsed:.1f}s")
else:
    print(f"✗ No tool call. Response: {msg.content[:100]}")


✓ Tool called: search_papers
  Arguments: {'query': 'reinforcement learning for robotics', 'max_results': 10}
  Time: 0.8s


### 4b. Tool Response Round-Trip

In [7]:
# Simulate the full tool call → tool result → final response cycle
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Find papers about neural network pruning"},
]

# Step 1: Get tool call
resp1 = call_llm(messages, tools=tools)
msg1 = resp1.choices[0].message

if msg1.tool_calls:
    tc = msg1.tool_calls[0]
    print(f"Step 1 — Tool call: {tc.function.name}({tc.function.arguments})")

    # Step 2: Simulate tool result
    tool_result = json.dumps([
        {"title": "Structured Pruning of Deep Neural Networks", "authors": "Li et al.", "year": 2025,
         "abstract": "We propose a novel structured pruning method that reduces model size by 80% with minimal accuracy loss."},
        {"title": "Dynamic Sparsity in Transformer Models", "authors": "Chen & Wang", "year": 2025,
         "abstract": "A runtime pruning strategy for transformer attention heads based on input complexity."}
    ])

    messages.append({"role": "assistant", "content": msg1.content, "tool_calls": [
        {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
    ]})
    messages.append({"role": "tool", "tool_call_id": tc.id, "content": tool_result})

    # Step 3: Get final response
    t0 = time.time()
    resp2 = call_llm(messages, tools=tools)
    elapsed = time.time() - t0
    final = resp2.choices[0].message.content or ""
    print(f"\nStep 2 — Final response ({elapsed:.1f}s):")
    print(final[:500])
else:
    print("No tool call made in step 1")


Step 1 — Tool call: search_papers({"query":"neural network pruning","max_results":10})



Step 2 — Final response (3.6s):
I've found a couple of recent papers on neural network pruning that might be of interest to you. Here are the details:

1. **Structured Pruning of Deep Neural Networks**  
   - **Authors**: Li et al.  
   - **Year**: 2025  
   - **Abstract**: We propose a novel structured pruning method that reduces model size by 80% with minimal accuracy loss.  

2. **Dynamic Sparsity in Transformer Models**  
   - **Authors**: Chen & Wang  
   - **Year**: 2025  
   - **Abstract**: A runtime pruning strategy fo


### 4c. Tool Restraint (should answer directly)

In [8]:
test_cases = [
    ("What is the capital of France?", False),
    ("What year did World War 2 end?", False),
    ("Search the web for quantum computing breakthroughs in 2026", True),
    ("Find me papers about attention mechanisms", True),
    ("Explain what a neural network is", False),
]

print(f"Testing tool restraint with {MODEL}:\n")
for query, expect_tool in test_cases:
    t0 = time.time()
    resp = call_llm(
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": query}],
        tools=tools
    )
    elapsed = time.time() - t0
    msg = resp.choices[0].message
    called = bool(msg.tool_calls)
    tool_name = msg.tool_calls[0].function.name if called else "-"
    status = "✓" if called == expect_tool else "✗"
    print(f"  {status} expect={expect_tool} got={called} tool={tool_name:20s} ({elapsed:.1f}s) | {query[:50]}")


Testing tool restraint with xai.grok-3-fast:



  ✓ expect=False got=False tool=-                    (0.4s) | What is the capital of France?


  ✓ expect=False got=False tool=-                    (1.2s) | What year did World War 2 end?


  ✓ expect=True got=True tool=search_tavily        (0.8s) | Search the web for quantum computing breakthroughs


  ✓ expect=True got=True tool=search_papers        (0.6s) | Find me papers about attention mechanisms


  ✓ expect=False got=False tool=-                    (12.4s) | Explain what a neural network is


## 5. Structured JSON Output (Entity Extraction)

In [9]:
t0 = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": '''Extract entities from the following text. Return ONLY a JSON array.

Text: "DeepMind published AlphaFold results at the NeurIPS conference in New Orleans"

Format: [{"name": "X", "type": "PERSON|PLACE|SYSTEM|EVENT", "description": "brief"}]
If no entities found: []'''}],
    max_tokens=400
)
elapsed = time.time() - t0
raw = resp.choices[0].message.content.strip()
print(f"Raw response ({elapsed:.1f}s):")
print(raw[:300])

try:
    start, end = raw.find("["), raw.rfind("]")
    if start >= 0 and end > start:
        entities = json.loads(raw[start:end+1])
        print(f"\nParsed {len(entities)} entities:")
        for e in entities:
            print(f"  • {e.get('name')} ({e.get('type')}): {e.get('description', '')}")
except json.JSONDecodeError as ex:
    print(f"\nJSON parse error: {ex}")


Raw response (1.2s):
[
    {"name": "DeepMind", "type": "SYSTEM", "description": "AI research organization"},
    {"name": "AlphaFold", "type": "SYSTEM", "description": "AI system for protein structure prediction"},
    {"name": "NeurIPS", "type": "EVENT", "description": "Conference on Neural Information Processing Syst

Parsed 4 entities:
  • DeepMind (SYSTEM): AI research organization
  • AlphaFold (SYSTEM): AI system for protein structure prediction
  • NeurIPS (EVENT): Conference on Neural Information Processing Systems
  • New Orleans (PLACE): City in Louisiana, USA


## 6. Context Engineering

In [10]:
MODEL_TOKEN_LIMITS = {
    "xai.grok-3-fast": 131072,
    "xai.grok-3-mini-fast": 131072,
    "qwen3:1.7b": 32768,
}

def calculate_context_usage(context: str, model: str = MODEL) -> dict:
    estimated_tokens = len(context) // 4
    max_tokens = MODEL_TOKEN_LIMITS.get(model, 131072)
    percentage = (estimated_tokens / max_tokens) * 100
    return {"tokens": estimated_tokens, "max": max_tokens, "percent": round(percentage, 1)}

# Test with growing context
for multiplier in [100, 1000, 5000, 20000]:
    ctx = "Sample research context about neural networks. " * multiplier
    usage = calculate_context_usage(ctx)
    print(f"  {multiplier:>5}x repeat → {usage['tokens']:>6} tokens / {usage['max']} ({usage['percent']}%)")

# Test LLM summarization
print("\nTesting summarization...")
t0 = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": """Summarize in 3 bullet points:

Reinforcement learning has transformed robotics in recent years. Key advances include sim-to-real
transfer learning, which allows policies trained in simulation to work on physical robots. Reward
shaping techniques have made training more sample-efficient. Multi-agent RL enables teams of robots
to coordinate on complex tasks like warehouse logistics."""}],
    max_tokens=300
)
elapsed = time.time() - t0
print(f"Summary ({elapsed:.1f}s):")
print(resp.choices[0].message.content)


    100x repeat →   1175 tokens / 131072 (0.9%)
   1000x repeat →  11750 tokens / 131072 (9.0%)
   5000x repeat →  58750 tokens / 131072 (44.8%)
  20000x repeat → 235000 tokens / 131072 (179.3%)

Testing summarization...


Summary (1.3s):
- Reinforcement learning has significantly advanced robotics, particularly through sim-to-real transfer learning, enabling policies trained in simulations to be effectively applied to physical robots.
- Reward shaping techniques in reinforcement learning have improved training efficiency, allowing robots to learn tasks with fewer samples.
- Multi-agent reinforcement learning facilitates coordination among teams of robots, enhancing their ability to perform complex tasks such as warehouse logistics.


## 7. Model Comparison Summary

Run this cell after testing with different `MODEL` values to compare results.
Change the `MODEL` variable in the Setup cell and re-run the notebook.

| Feature | xai.grok-3-fast | xai.grok-3-mini-fast | qwen3:1.7b (Ollama) |
|---|---|---|---|
| Hosting | OCI GenAI (cloud) | OCI GenAI (cloud) | Local (Codespace) |
| Auth | OCI CLI profile | OCI CLI profile | None needed |
| Speed | ~0.5-1.5s | ~2-6s | ~1-3s |
| Tool calling | Excellent | Good | Good (with restraint prompt) |
| JSON output | Excellent | Good | Good |
| Cost | Pay-per-use | Pay-per-use | Free (CPU time) |
| Offline | No | No | Yes |


## Cleanup

In [11]:
try:
    vector_conn.cursor().execute(f"DROP TABLE {TEST_TABLE} PURGE")
    vector_conn.commit()
    print(f"Dropped {TEST_TABLE} ✓")
except:
    pass
vector_conn.close()
print("Oracle connection closed ✓")
print("\n✅ All tests complete!")


Dropped OCI_GENAI_TEST_VECTORS ✓
Oracle connection closed ✓

✅ All tests complete!
